In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import librosa
from transformers import Wav2Vec2Model, Wav2Vec2Processor
from pathlib import Path
from tqdm import tqdm

In [ ]:
import torch
import librosa
from transformers import Wav2Vec2Processor, Wav2Vec2Model

# 1. Load the pre-trained foundational encoder
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-large-xlsr-53")
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-large-xlsr-53")
model.eval()

# 2. Ingest your pristine 300ms steady-state vowel clip
# (300ms at 16kHz sample rate = 4800 audio samples)
audio_segment, sr = librosa.load("data/processed/M062_Mic_a_non-resonant_1st.wav", sr=16000)

# 3. Format input tensor maps
inputs = processor(audio_segment, sampling_rate=16000, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
    
    # Extract the hidden states from the final transformer context layer
    # Output shape will be exactly: [Batch=1, Time_Steps=15, Hidden_Dim=1024]
    latent_embeddings = outputs.last_hidden_state
    
    # Apply temporal average pooling to create a 1D signature vector
    # Resulting shape: [1024]
    vocal_signature = torch.mean(latent_embeddings, dim=1).squeeze()